Cell 1 Env Setup

In [251]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import pickle
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Ensure reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("✅ Environment ready.")

✅ Environment ready.


Cell 2 Loading Dataset

In [252]:
# Load the datasets
v_df = pd.read_csv('BCCC-VolSCs-2023_Vulnerable.csv')
s_df = pd.read_csv('BCCC-VolSCs-2023_Secure.csv')

# Combine and shuffle
df = pd.concat([v_df, s_df], axis=0).reset_index(drop=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Separate feature types
opcode_cols = [c for c in df.columns if 'Opcode weight' in c]
metadata = ['Unnamed: 0', 'hash_id', 'ast_id', 'ast_src', 'ast_nodetype', 'label']
static_cols = [c for c in df.columns if c not in opcode_cols and c not in metadata]

# FORCE EXACT DIMENSIONS (Matching the logic for 138 and 56)
# We select the top most frequent features to ensure quality
final_opcode_features = (df[opcode_cols] != 0).sum().sort_values(ascending=False).head(138).index.tolist()
final_static_features = (df[static_cols] != 0).sum().sort_values(ascending=False).head(56).index.tolist()

X_static = df[final_static_features].fillna(0).values
X_opcode = df[final_opcode_features].fillna(0).values
y = df['label'].values

print(f"📊 Features Selected -> Static: {len(final_static_features)} | Opcode: {len(final_opcode_features)}")

📊 Features Selected -> Static: 56 | Opcode: 138


Cell 3

In [253]:
# Train/Test Split
s_train, s_test, o_train, o_test, y_train, y_test = train_test_split(
    X_static, X_opcode, y, test_size=0.2, random_state=42, stratify=y
)

# Scaling
sc_static = StandardScaler()
sc_opcode = StandardScaler()

s_train_scaled = sc_static.fit_transform(s_train)
s_test_scaled = sc_static.transform(s_test)
o_train_scaled = sc_opcode.fit_transform(o_train)
o_test_scaled = sc_opcode.transform(o_test)

# --- SAVE DEPLOYMENT ARTIFACTS ---
with open('scaler_static.pkl', 'wb') as f: pickle.dump(sc_static, f)
with open('scaler_opcode.pkl', 'wb') as f: pickle.dump(sc_opcode, f)

manifest = {
    'static_names': final_static_features,
    'opcode_names': final_opcode_features
}
with open('manifest.pkl', 'wb') as f:
    pickle.dump(manifest, f)

# Convert to Tensors
train_s = torch.tensor(s_train_scaled, dtype=torch.float32)
train_o = torch.tensor(o_train_scaled, dtype=torch.float32)
train_y = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
test_s = torch.tensor(s_test_scaled, dtype=torch.float32)
test_o = torch.tensor(o_test_scaled, dtype=torch.float32)

print("✅ Scalers and Manifest saved. Tensors initialized.")

✅ Scalers and Manifest saved. Tensors initialized.


Cell 4 Model Architecture (The Fixed Structure)

In [254]:
class GDSAN_Hybrid(nn.Module):
    def __init__(self, static_dim, opcode_dim):
        super(GDSAN_Hybrid, self).__init__()
        self.static_fc1 = nn.Linear(static_dim, 128)
        self.static_gate = nn.Linear(static_dim, 128)
        self.static_fc2 = nn.Linear(128, 64)
        
        self.opcode_projection = nn.Linear(opcode_dim, 64)
        self.multihead_attn = nn.MultiheadAttention(embed_dim=64, num_heads=8, batch_first=True)
        self.attn_norm = nn.LayerNorm(64)
        
        self.fusion_layer = nn.Sequential(
            nn.Linear(64 + 64, 64),
            nn.BatchNorm1d(64), 
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, s, o):
        gate = torch.sigmoid(self.static_gate(s))
        s_feat = F.relu(self.static_fc1(s)) * gate
        s_feat = self.static_fc2(s_feat)
        
        o_proj = self.opcode_projection(o).unsqueeze(1)
        attn_out, _ = self.multihead_attn(o_proj, o_proj, o_proj)
        o_feat = self.attn_norm(attn_out.squeeze(1) + o_proj.squeeze(1))
        
        combined = torch.cat((s_feat, o_feat), dim=1)
        return self.fusion_layer(combined)

model = GDSAN_Hybrid(len(final_static_features), len(final_opcode_features))
print("✅ Model Architecture Initialized.")

✅ Model Architecture Initialized.


Cell 5 Training and Final Saving

In [255]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCELoss()

# Training loop
epochs = 50 
model.train()
for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(train_s, train_o)
    loss = criterion(outputs, train_y)
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

# Final Evaluation
model.eval()
with torch.no_grad():
    y_probs = model(test_s, test_o).numpy().flatten()
    y_pred = (y_probs > 0.5).astype(int)
    
acc = accuracy_score(y_test, y_pred)
print(f"🚀 Final Test Accuracy: {acc:.4f}")

# Save the weights
torch.save(model.state_dict(), 'gdsan_model.pth')
print("✅ Model weights saved as 'gdsan_model.pth'.")

Epoch [10/50], Loss: 0.5971
Epoch [20/50], Loss: 0.5566
Epoch [30/50], Loss: 0.5433
Epoch [40/50], Loss: 0.5317
Epoch [50/50], Loss: 0.5185
🚀 Final Test Accuracy: 0.7560
✅ Model weights saved as 'gdsan_model.pth'.
